# Text Representation

Text representation in NLP means converting text into a numerical form that a machine learning model can understand and process.

ALI ZUHAIR ALSAFFAR 2240005706

## What is Embedding?

Embeddings in NLP is a technique where individual words are represented as real-valued vectors and captures inter-word semantics.


In this notebook, We going to intreduce 2 techniques for embedding. These techniques will be used for a machine learning models such as SVM, Random forest, ... ect.

<img src="https://drive.google.com/uc?export=view&id=1jd0u_sGppDKqBYbhtJfGxp14lnUWUTTw" width="900">

# 1- TF-IDF

<img src="https://drive.google.com/uc?export=view&id=1saSt0mMgOQ2ybbTms1lu_ruhUcBWkKzL" width="500">

TF-IDF stands for term frequency-inverse document frequency. It is a measure that discounts common words. Used in the fields of information retrieval (IR) and machine learning, that can quantify the importance of words in a document amongst a collection of documents (also known as a corpus).

### Components of TF-IDF

1. **TF (Term Frequency):**  
   Measures how frequently a term appears in a document.


$$ \text{TF}(t, d) = \frac{\text{Number of times term } t \text{ appears in document } d}{\text{Total number of terms in document } d} $$


2. **IDF (Inverse Document Frequency):**  
   Measures how important a term is across all documents. Words that appear in many documents get lower scores.

$$    \text{IDF}(t) = \log \left(\frac{\text{Number of all documents N}}{\text{Number of documents containing the term } t}\right) $$

<img src="https://drive.google.com/uc?export=view&id=1JqPILC8TTh3yDQZCSuPNXwm6ER5YeJFh" width="900">

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
doc_1 = "Data is the oil of the digital economy"
doc_2 = "Data is a new oil"

data = [doc_1, doc_2]


In [3]:
tfidf = TfidfVectorizer()
result = tfidf.fit_transform(data) # returns sparce matrix

In [4]:
import pandas as pd
df = pd.DataFrame(result.toarray(), columns=tfidf.get_feature_names_out())
df

,data,digital,economy,is,new,of,oil,the
0,0.243777,0.34262,0.34262,0.243777,0.000000,0.34262,0.243777,0.68524
1,0.448321,0.00000,0.00000,0.448321,0.630099,0.00000,0.448321,0.00000


# Cosine similarity

In NLP, Cosine similarity is a metric used to measure how similar the documents are.


$$ \text{cosine similarity} = \frac{A \cdot B}{\|A\| \times \|B\|} $$

Where:  
$ A \cdot B $  = dot product of vectors A and B  
$ \|A\| $ = magnitude (length) of vector A  
$ \|B\| $ =  magnitude (length) of vector B

#### Intuition

- If vectors point in the **same direction**, cosine similarity = **1** (maximum similarity).
- If vectors are **orthogonal (90° apart)**, cosine similarity = **0** (no similarity).

<img src="https://drive.google.com/uc?export=view&id=1b-o8CVfHBsjUGY_hXmxevU91gHidIuxO" width="900">

In [5]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
# Sample text data (replace with your own documents)

doc_1 = "Data is the oil of the digital economy"
doc_2 = "Data is a new oil"

data = [doc_1, doc_2]

In [7]:
tfidf_vectorizer = TfidfVectorizer() # Create a CountVectorizer instance
vector_matrix = tfidf_vectorizer.fit_transform(data) # Fit and transform the documents into numerical vectors

In [8]:
# Calculate the cosine similarity between the documents
cosine_similarity_matrix = cosine_similarity(vector_matrix)

df_cosine = pd.DataFrame(data=cosine_similarity_matrix, index=data, columns=data)

df_cosine

,Data is the oil of the digital economy,Data is a new oil
Data is the oil of the digital economy,1.000000,0.327871
Data is a new oil,0.327871,1.000000


# 2- What is Word2vec?

Word2Vec consists of models for generating word embedding. These models are two-layer neural networks having one input layer, one hidden layer, and one output layer.

Word2Vec utilizes two architectures :
1. CBOW (Continuous Bag of Words)
2. **Skip Gram**
<img src="https://drive.google.com/uc?export=view&id=1K38nEu_KhgtSJuG2RySwc6U5AAjVAgWZ" width="600">


Run this command in terminal to install
> pip install gensim

We will use fake and real news dataset to do our expirament. You can find the dataset here: https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset?select=True.csv


In [9]:
import pandas as pd
import nltk
import numpy as np
import gensim
from nltk.tokenize import word_tokenize

# IMPORTANT FIX: word_tokenize() needs NLTK's 'punkt' tokenizer data downloaded first.
# The notebook previously called word_tokenize() further below without ever downloading it,
# which raises a LookupError and stops execution of every cell after it (including all Task cells).
nltk.download('punkt')
try:
    nltk.download('punkt_tab')  # required on newer NLTK versions (>=3.8.2); safe to skip on older ones
except Exception:
    pass

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\aliza\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\aliza\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [10]:
# IMPORTANT FIX: '/content/True.csv' only exists inside Google Colab after you've uploaded the file there.
# Outside Colab (or if you haven't uploaded it), this raised FileNotFoundError and silently killed
# every cell below it in a 'Run All'. We now load it safely so the rest of the notebook still runs
# even if this particular demo dataset isn't available.
NEWS_DATASET_PATH = "True.csv"   # update this path if your file is elsewhere

try:
    df = pd.read_csv(NEWS_DATASET_PATH, engine="python", on_bad_lines="skip")
    print(f"Dataset loaded successfully from '{NEWS_DATASET_PATH}'. Shape: {df.shape}")
    display(df.head())
except FileNotFoundError:
    df = None
    print(f"Could not find '{NEWS_DATASET_PATH}' -- skipping this demo section.\n"
          "(This only affects the Word2Vec walkthrough example above the Tasks; "
          "it does not affect Tasks 1-5 below.)\n"
          "To run this demo, download the dataset from:\n"
          "https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset\n"
          "and place True.csv in the same folder as this notebook.")

Could not find 'True.csv' -- skipping this demo section.
(This only affects the Word2Vec walkthrough example above the Tasks; it does not affect Tasks 1-5 below.)
To run this demo, download the dataset from:
https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset
and place True.csv in the same folder as this notebook.


In [11]:
if df is not None:
    tokens = []

    for i in df['text']:
        token = word_tokenize(i)
        tokens.append(token)

In [12]:
if df is not None:
    w2v = gensim.models.Word2Vec(tokens, min_count=1, vector_size=100, window=5, sg=1)

In [13]:
if df is not None:
    print("Cosine similarity between 'provide' and 'program' - Skip Gram : ", w2v.wv.similarity('provide', 'program'))

In [14]:
if df is not None:
    print("words that similar to 'program' - Skip Gram : ", w2v.wv.most_similar('program'))

# Tasks

### Task 1: Cosine Similarity
Use the Cosine Similarity method to determine how similar the following sentences are.

'This is the first document.',  
'This document is the second document.',  
'And this is the third one.',  
'Is this the first document?'  


In [15]:
# The 4 sentences to compare
task1_sentences = [
    'This is the first document.',
    'This document is the second document.',
    'And this is the third one.',
    'Is this the first document?'
]

# Convert the sentences into TF-IDF vectors
task1_vectorizer = TfidfVectorizer()
task1_vectors = task1_vectorizer.fit_transform(task1_sentences)

# Compute pairwise cosine similarity between every pair of sentences
task1_cosine_sim = cosine_similarity(task1_vectors)

# Display as a labeled similarity matrix
task1_df = pd.DataFrame(task1_cosine_sim, index=task1_sentences, columns=task1_sentences)
task1_df

,This is the first document.,This document is the second document.,And this is the third one.,Is this the first document?
This is the first document.,1.000000,0.646926,0.307772,1.000000
This document is the second document.,0.646926,1.000000,0.225240,0.646926
And this is the third one.,0.307772,0.225240,1.000000,0.307772
Is this the first document?,1.000000,0.646926,0.307772,1.000000


### Task 2: TF-IDF
Use tf-idf method on the sentences below to determine the important words.

'data science is one of the most important fields of science',  
'this is one of the best data science courses',  
'data scientists analyze data'  


In [16]:
# The 3 sentences to analyze
task2_sentences = [
    'data science is one of the most important fields of science',
    'this is one of the best data science courses',
    'data scientists analyze data'
]

# Compute TF-IDF weights for every word in every sentence
task2_vectorizer = TfidfVectorizer()
task2_matrix = task2_vectorizer.fit_transform(task2_sentences)

task2_df = pd.DataFrame(
    task2_matrix.toarray(),
    columns=task2_vectorizer.get_feature_names_out(),
    index=[f"Sentence {i+1}" for i in range(len(task2_sentences))]
)
task2_df

,analyze,best,courses,data,fields,important,is,most,of,one,science,scientists,the,this
Sentence 1,0.000000,0.000000,0.000000,0.189526,0.320895,0.320895,0.244049,0.320895,0.488098,0.244049,0.488098,0.000000,0.244049,0.000000
Sentence 2,0.000000,0.400294,0.400294,0.236420,0.000000,0.000000,0.304434,0.000000,0.304434,0.304434,0.304434,0.000000,0.304434,0.400294
Sentence 3,0.542701,0.000000,0.000000,0.641055,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.542701,0.000000,0.000000


In [17]:
# The most important word(s) in each sentence = the word(s) with the highest TF-IDF weight in that row
for i, sentence in enumerate(task2_sentences):
    row = task2_df.iloc[i]
    top_word = row.idxmax()
    print(f"Sentence {i+1}: \"{sentence}\"")
    print(f"  -> Most important word: '{top_word}' (TF-IDF = {row[top_word]:.3f})\n")

Sentence 1: "data science is one of the most important fields of science"
  -> Most important word: 'of' (TF-IDF = 0.488)

Sentence 2: "this is one of the best data science courses"
  -> Most important word: 'best' (TF-IDF = 0.400)

Sentence 3: "data scientists analyze data"
  -> Most important word: 'data' (TF-IDF = 0.641)



## Word2vec

### Task 3:

Download the Simpsons dataset **(simpsons_script_lines.csv)** and apply the preprocessing procedure.  
Use the **'spoken_words'** column.
```
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[0-9]", '', text)
    text = re.sub(r"[)(,”“.’$-]", '', text)
    return text
```
Create a skip gram Word2Vec model as below.
```
Skip_gram_model = gensim.models.Word2Vec(tokens, min_count = 1, vector_size = 100, window = 5, sg = 1)

In [18]:
import re
import glob

# IMPORTANT FIX: auto-detect the Simpsons CSV instead of requiring an exact filename.
# Downloaded/uploaded copies are often named things like 'simpsons_script_lines.csv',
# 'simpsons_script_lines__1_.csv', etc. -- we search for any file matching that pattern
# in the current folder so a filename mismatch can't silently break this section.
matches = glob.glob("simpsons_script_lines*.csv") + glob.glob("*simpsons*.csv")
matches = list(dict.fromkeys(matches))  # de-duplicate while preserving order

if matches:
    SIMPSONS_PATH = matches[0]
    simpsons_df = pd.read_csv(SIMPSONS_PATH, low_memory=False)
    print(f"Dataset loaded successfully from '{SIMPSONS_PATH}'. Shape: {simpsons_df.shape}")
else:
    simpsons_df = None
    print("Could not find a Simpsons CSV file (looked for any file matching 'simpsons*.csv').\n"
          "Please make sure the dataset file is in the same folder as this notebook.")

Dataset loaded successfully from 'simpsons_script_lines (1).csv'. Shape: (158314, 2)


In [19]:
if simpsons_df is not None:
    print("Columns:", list(simpsons_df.columns))
    display(simpsons_df.head())

    # Drop rows with no spoken_words
    simpsons_df = simpsons_df.dropna(subset=["spoken_words"]).reset_index(drop=True)
    print(f"Rows with spoken text: {len(simpsons_df)}")

Columns: ['raw_character_text', 'spoken_words']


,raw_character_text,spoken_words
0,Miss Hoover,"No, actually, it was a little of both. Sometim..."
1,Lisa Simpson,Where's Mr. Bergstrom?
2,Miss Hoover,I don't know. Although I'd sure like to talk t...
3,Lisa Simpson,That life is worth living.
4,Edna Krabappel-Flanders,The polls will be open from now until the end ...


Rows with spoken text: 131855


**Pre-processing** — apply the `clean_text` function given in the task (lowercase, remove digits, remove punctuation), then tokenize each line into words.

In [20]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[0-9]", '', text)
    text = re.sub(r"[)(,”“.’$-]", '', text)
    return text

if simpsons_df is not None:
    simpsons_df["clean_text"] = simpsons_df["spoken_words"].apply(clean_text)
    display(simpsons_df[["spoken_words", "clean_text"]].head())

,spoken_words,clean_text
0,"No, actually, it was a little of both. Sometim...",no actually it was a little of both sometimes ...
1,Where's Mr. Bergstrom?,where's mr bergstrom?
2,I don't know. Although I'd sure like to talk t...,i don't know although i'd sure like to talk to...
3,That life is worth living.,that life is worth living
4,The polls will be open from now until the end ...,the polls will be open from now until the end ...


In [21]:
if simpsons_df is not None:
    tokens = []
    for line in simpsons_df["clean_text"]:
        token = word_tokenize(line)
        tokens.append(token)

    print(f"Number of tokenized lines: {len(tokens)}")
    print("Example tokenized line:", tokens[0] if tokens else None)

Number of tokenized lines: 131855
Example tokenized line: ['no', 'actually', 'it', 'was', 'a', 'little', 'of', 'both', 'sometimes', 'when', 'a', 'disease', 'is', 'in', 'all', 'the', 'magazines', 'and', 'all', 'the', 'news', 'shows', 'it', "'s", 'only', 'natural', 'that', 'you', 'think', 'you', 'have', 'it']


**Train a Skip-gram Word2Vec model** on the tokenized lines (`sg=1` selects Skip-gram over CBOW).

In [22]:
if simpsons_df is not None:
    Skip_gram_model = gensim.models.Word2Vec(tokens, min_count=1, vector_size=100, window=5, sg=1)
    print("Skip-gram Word2Vec model trained.")
    print("Vocabulary size:", len(Skip_gram_model.wv.key_to_index))

Skip-gram Word2Vec model trained.
Vocabulary size: 44282


### Task 4

Use: wv.most_similar() method to :

1.	Find the words similar to “homer”.

2.	Find the words similar to “marge”.

3. Find the words similar to “bart”



In [23]:
if simpsons_df is not None:
    for name in ["homer", "marge", "bart"]:
        print(f"Words similar to '{name}':")
        try:
            for similar_word, score in Skip_gram_model.wv.most_similar(name):
                print(f"  {similar_word}  (similarity: {score:.3f})")
        except KeyError:
            print(f"  '{name}' was not found in the model's vocabulary.")
        print()

Words similar to 'homer':
  abe  (similarity: 0.887)
  marge  (similarity: 0.850)
  grampa  (similarity: 0.824)
  bart  (similarity: 0.824)
  millionaire  (similarity: 0.816)
  ned  (similarity: 0.808)
  superstar  (similarity: 0.805)
  mister  (similarity: 0.804)
  bartholomew  (similarity: 0.802)
  eliza  (similarity: 0.801)

Words similar to 'marge':
  abe  (similarity: 0.869)
  homer  (similarity: 0.850)
  lisa  (similarity: 0.842)
  sweetie  (similarity: 0.839)
  sweetheart  (similarity: 0.832)
  honey  (similarity: 0.829)
  becky  (similarity: 0.827)
  allison  (similarity: 0.824)
  lis  (similarity: 0.817)
  carefully  (similarity: 0.816)

Words similar to 'bart':
  milhouse  (similarity: 0.889)
  lisa  (similarity: 0.874)
  jessica  (similarity: 0.858)
  grampa  (similarity: 0.849)
  abe  (similarity: 0.845)
  allison  (similarity: 0.839)
  sweetie  (similarity: 0.833)
  eliza  (similarity: 0.830)
  stampy  (similarity: 0.830)
  luann  (similarity: 0.829)



### Task 5

Use the wv.doesnt_match() method to :

1.	Find which of 'jimbo', 'milhouse’, and 'kearney’ does not belong to the list.

3.	Find the odd one among "nelson", "bart", and "milhouse".

4.	Find the odd one among ‘homer', 'patty', and ‘selma'.

*Hint: You need to pass the strings as List*

In [24]:
if simpsons_df is not None:
    word_groups = [
        ['jimbo', 'milhouse', 'kearney'],
        ['nelson', 'bart', 'milhouse'],
        ['homer', 'patty', 'selma'],
    ]

    for group in word_groups:
        try:
            odd_one_out = Skip_gram_model.wv.doesnt_match(group)
            print(f"{group} -> odd one out: '{odd_one_out}'")
        except KeyError as e:
            print(f"{group} -> could not compare, missing word in vocabulary: {e}")

['jimbo', 'milhouse', 'kearney'] -> odd one out: 'milhouse'
['nelson', 'bart', 'milhouse'] -> odd one out: 'nelson'
['homer', 'patty', 'selma'] -> odd one out: 'homer'
